# EN3150 Pattern Recognition - Assignment 03
## Resource-Constrained CNN for Edge Image Classification

**Group :** GalTricals

**Group Members**
- CHINTHAKA M.K.M.G. - 230105H
- DILMITH GS - 230149U
- KAUSHAL S.V. - 230326K
- RAJAPAKSHA J.S.W. - 230511A


###  Introduction and Objectives
The objective of this project is to design and evaluate a lightweight Convolutional Neural Network (CNN) constrained to under 100,000 trainable parameters. Our model is intended for deployment on a low resource edge device, specifically an **Automated Agricultural Sorting Machine** designed to classify 5 varieties of rice grains in real time.


To emulate the memory and computational constraints of edge devices (such as microcontrollers), the input images are downscaled to a maximum resolution of `64x64` pixels.


### 1. Data Preparation Strategy
We utilized the **Rice Image Dataset**, which originally contains 75,000 images across 5 classes.
To optimize training times and prevent Out Of Memory (OOM) failures typical in local environments, we randomly sub sampled exactly **2,000 images per class** (10,000 images total).

To prevent data leakage and ensure balanced classes, we applied a strict **Stratified Train ,Validation ,Test Split**:
* **Training Set:** 70% (7,000 images)
* **Validation Set:** 15% (1,500 images)
* **Testing Set:** 15% (1,500 images)

In [ ]:
# Setup Kaggle API and Download Rice Image Dataset
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Downloading Rice Image Dataset...")
!kaggle datasets download -d muratkokludataset/rice-image-dataset

print("Unzipping...")
!unzip -q rice-image-dataset.zip -d rice_dataset
print("Dataset Successfully Downloaded and Extracted!")



In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import os
import time
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)

# Locate the dataset directories
base_extract_dir = pathlib.Path('rice_dataset')
target_dirs = list(base_extract_dir.rglob('Arborio'))
data_dir = target_dirs[0].parent if target_dirs else base_extract_dir

class_names = np.array(sorted([item.name for item in data_dir.glob('*') if item.is_dir()]))
print(f"Classes: {class_names}")

# 1. Sub-sample 2,000 images per class
SAMPLES_PER_CLASS = 2000
file_paths = []
labels = []

for i, class_name in enumerate(class_names):
    class_dir = data_dir / class_name
    class_files = list(class_dir.glob('*.jpg'))[:SAMPLES_PER_CLASS]
    file_paths.extend([str(path) for path in class_files])
    labels.extend([i] * len(class_files))

file_paths = np.array(file_paths)
labels = np.array(labels)

print(f"Total sub-sampled images: {len(file_paths)}")

# 2. Stratified Split (70% Train, 15% Val, 15% Test)
X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    file_paths, labels, test_size=0.30, stratify=labels, random_state=42)

X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(f"Training samples: {len(X_train_paths)}")
print(f"Validation samples: {len(X_val_paths)}")
print(f"Testing samples: {len(X_test_paths)}")

# 3. Create High-Performance tf.data.Datasets
IMG_SIZE = (64, 64)
BATCH_SIZE = 32

def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = tf.data.Dataset.from_tensor_slices((X_train_paths, y_train)).map(process_path, num_parallel_calls=AUTOTUNE).shuffle(10000).batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val_paths, y_val)).map(process_path, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test_paths, y_test)).map(process_path, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

# Visualize Samples
plt.figure(figsize=(15, 6))
for images, batch_labels in train_dataset.take(1):
    for i in range(10):
        if i < len(images):
            ax = plt.subplot(2, 5, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(class_names[batch_labels[i]])
            plt.axis("off")
plt.suptitle("Sample Rice Grains (Downscaled to 64x64)", fontsize=16)
plt.tight_layout()
plt.show()



## 2. Custom Architecture Design & Theoretical Justifications

#### 2.1. Activation Functions & The Vanishing Gradient Problem
As discussed in our Pattern Recognition lectures (L06), deep neural networks utilizing standard `Sigmoid` or `Tanh` activation functions frequently encounter the **Vanishing Gradient** problem. During backpropagation, gradients at the tails of these functions approach zero, effectively halting weight updates in earlier layers.

To counteract this, both our architectures exclusively use **ReLU (Rectified Linear Unit)** activations:
$$f(x) = \max(0, x)$$
ReLU does not saturate for positive values (its derivative is exactly 1 for x > 0). Furthermore, its simple thresholding nature makes it highly computationally efficient, making it ideal for low-power edge devices.

#### 2.2. Model A: Standard CNN Architecture
Model A serves as our high-capacity baseline, interleaving standard `Conv2D` and `MaxPooling2D` layers.
The trainable parameters for a standard 2D Convolution layer are calculated as:
$$Parameters = (K_h \times K_w \times C_{in} + 1) \times C_{out}$$
*(Where $K_h, K_w$ are kernel dimensions, $C_{in}$ is input channels, and $C_{out}$ is output filters)*

**Calculation Example (Layer 1):** 
Given a 3x3 kernel, 3 input channels (RGB), and 32 output filters:
$$Params = (3 \times 3 \times 3 + 1) \times 32 = 896$$

In total, Model A contains **1,142,597 trainable parameters**, resulting in an estimated disk size of **~13.44 MB**.


In [ ]:
from tensorflow.keras import layers, models

def build_model_a(input_shape=(64, 64, 3), num_classes=5):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(1./255), # Normalize to [0, 1] natively
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ], name="Model_A_Standard_CNN")
    return model

model_a = build_model_a(num_classes=len(class_names))
print("="*50)
model_a.summary()



#### 2.3. Model B: Lightweight Edge-Optimized CNN
To strictly remain under the 100,000 parameter constraint, Model B replaces standard convolutions with **Depthwise Separable Convolutions**. This operation splits the convolution into a spatial depthwise step and a channel-wise pointwise (1x1) step, drastically reducing Multiply-Accumulate (MAC) operations.

The parameters for a Depthwise Separable Convolution are calculated as:
$$Parameters = (K_h \times K_w \times C_{in}) + (1 \times 1 \times C_{in} \times C_{out} + C_{out})$$

**Calculation Example (Layer 1):**
$$Params = (3 \times 3 \times 3) + (1 \times 1 \times 3 \times 32 + 32) = 27 + 128 = 155$$
This represents a massive **82.7% reduction** in parameters for the first layer compared to Model A.

**Constraint Verification:** 
Our final compiled Model B contains **79,168 trainable parameters**, successfully satisfying the `< 100,000` assignment constraint. Its size on disk is merely **0.99 MB**.


In [ ]:
def build_model_b(input_shape=(64, 64, 3), num_classes=5):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(1./255), # Normalize to [0, 1] natively
        layers.SeparableConv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.SeparableConv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.SeparableConv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.SeparableConv2D(256, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ], name="Model_B_Lightweight_CNN")
    return model

model_b = build_model_b(num_classes=len(class_names))
print("="*50)
model_b.summary()



## 3. Optimizer Selection & Tuning
As established in our Deep Learning modules, optimization algorithms dictate how weights are updated during backpropagation. We evaluate 5 different optimizers:

1. **SGD (Stochastic Gradient Descent):** Updates parameters using the gradient of a single batch. Can easily get stuck in local minima or saddle points, causing slow convergence.
2. **SGD + Momentum:** Adds a velocity term to dampen oscillations and accelerate convergence past shallow local minima.
3. **RMSprop (Root Mean Square Propagation):** Addresses infinitesimally small learning rates by focusing on recent gradients and decaying older ones.
4. **Adagrad:** Adapts per-parameter learning rates, but can prematurely halt learning due to monotonic learning rate decay.
5. **Adam (Adaptive Moment Estimation):** Combines the benefits of both Momentum and RMSprop. It is highly robust, forgives poor hyperparameters, and typically achieves faster initial reductions in training error.



In [ ]:
EPOCHS_TUNE = 15
learning_rate = 0.001

optimizers = {
    'SGD': tf.keras.optimizers.SGD(learning_rate=learning_rate),
    'SGD_Momentum': tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9),
    'Adam': tf.keras.optimizers.Adam(learning_rate=learning_rate),
    'RMSprop': tf.keras.optimizers.RMSprop(learning_rate=learning_rate),
    'Adagrad': tf.keras.optimizers.Adagrad(learning_rate=learning_rate)
}
history_dict = {}

print("Comparing 5 Optimizers on Model B...")
for opt_name, opt in optimizers.items():
    print(f"\n--- Training with {opt_name} ---")
    temp_model = tf.keras.models.clone_model(model_b)
    temp_model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = temp_model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS_TUNE, verbose=0)
    history_dict[opt_name] = history.history
    print(f"{opt_name} Final Val Accuracy: {history.history['val_accuracy'][-1]:.4f}")

# Plot Convergence
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
for opt_name in optimizers.keys():
    plt.plot(history_dict[opt_name]['val_loss'], label=opt_name, marker='o')
plt.title('Validation Loss Comparison (5 Optimizers)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True, alpha=0.6)

plt.subplot(1, 2, 2)
for opt_name in optimizers.keys():
    plt.plot(history_dict[opt_name]['val_accuracy'], label=opt_name, marker='^')
plt.title('Validation Accuracy Comparison (5 Optimizers)')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True, alpha=0.6)
plt.tight_layout()
plt.show()



## 4. Custom Model Training & Evaluation
We trained both Model A and Model B for **20 epochs** using the highest performing optimizer (Adam).

#### 4.1 Custom Models Comparison Table
| Metric | Model A (Standard CNN) | Model B (Lightweight CNN) |
| :--- | :--- | :--- |
| **Total Parameters** | 1,142,597 | 79,168 (< 100k constraint) |
| **Model Size on Disk** | 13,440 KB (~13.4 MB) | 994 KB (~0.99 MB) |
| **Training Time per Epoch** | ~76.2 seconds | ~52.0 seconds |
| **Test Accuracy** | 96.73% | 96.73% |

#### 4.2 Trade-offs and Overfitting Analysis
Upon analyzing the training results, a significant architectural trade-off is evident:

1. **Overfitting in Standard Convolutions:** **Model A** exhibits classic overfitting. While its training loss approaches zero, its validation loss begins to diverge and increase after epoch 5. Its massive parameter count (~1.14M) simply memorizes the training data.
2. **Natural Regularization:** By moving to Depthwise Separable Convolutions, **Model B** avoids overfitting completely. Its validation loss smoothly tracks the training loss. The constrained parameter space acts as a *natural regularizer*, forcing the network to learn generalized features rather than noise.
3. **Efficiency vs Performance:** Moving to depthwise separable convolutions reduced the parameter count by ~93% and the disk size from 13.4 MB to under 1 MB, *without losing any classification accuracy* (both achieved 96.73%). Furthermore, the training time per epoch dropped from 76 seconds to 52 seconds. 


In [ ]:
EPOCHS = 20 # Rubric Requirement
model_a.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_b.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("TRAINING MODEL A (STANDARD CNN)...")
t0 = time.time()
hist_a = model_a.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS, verbose=1)
time_a = (time.time() - t0)/EPOCHS

print("\nTRAINING MODEL B (LIGHTWEIGHT CNN)...")
t0 = time.time()
hist_b = model_b.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS, verbose=1)
time_b = (time.time() - t0)/EPOCHS

model_a.save('model_a.h5'); model_b.save('model_b.h5')
size_a = os.path.getsize('model_a.h5') / 1024
size_b = os.path.getsize('model_b.h5') / 1024

print(f"\nModel A Size: {size_a:.2f} KB | Avg Time/Ep: {time_a:.2f}s")
print(f"Model B Size: {size_b:.2f} KB | Avg Time/Ep: {time_b:.2f}s")

# Plot Loss Curves
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.plot(hist_a.history['loss'], label='Train Loss A', marker='o')
plt.plot(hist_a.history['val_loss'], label='Val Loss A', marker='o')
plt.title('Model A: Loss Curves'); plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(hist_b.history['loss'], label='Train Loss B', marker='o')
plt.plot(hist_b.history['val_loss'], label='Val Loss B', marker='o')
plt.title('Model B: Loss Curves'); plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
plt.show()



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("Evaluating on Test Set (Memory Optimized)...")

# Retrieve true labels from the manually split test set
# y_test array is strictly aligned with X_test_paths
test_labels = y_test

# Predict batch-by-batch
preds_a = np.argmax(model_a.predict(test_dataset, verbose=1), axis=1)
preds_b = np.argmax(model_b.predict(test_dataset, verbose=1), axis=1)

print("="*50)
print("MODEL A METRICS")
print("="*50)
print(classification_report(test_labels, preds_a, target_names=class_names))

print("="*50)
print("MODEL B METRICS")
print("="*50)
print(classification_report(test_labels, preds_b, target_names=class_names))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(test_labels, preds_a), annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Model A: Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual')

plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(test_labels, preds_b), annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('Model B: Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()



## 5. SOTA Fine-Tuning & Evaluation
We fine-tune **MobileNetV2** and **EfficientNetB0**.

**Normalization Handling:**
- `MobileNetV2` expects inputs in `[-1, 1]`. We employ a `Rescaling(1./127.5, offset=-1)` layer as the entry point.
- `EfficientNetB0` natively expects inputs in `[0, 255]` and includes an inherent computational normalization layer in its architecture. Therefore, we pass our raw `[0, 255]` pixel images directly into it to avoid degrading its pre-trained weights.



In [ ]:
print("FINE-TUNING SOTA MODELS...")
EPOCHS_SOTA = 15 # Transfer learning converges rapidly

# MobileNetV2 (Needs [-1, 1])
base_mn = tf.keras.applications.MobileNetV2(input_shape=(64, 64, 3), include_top=False, weights='imagenet')
base_mn.trainable = False
sota_mn = models.Sequential([
    layers.Input(shape=(64, 64, 3)),
    layers.Rescaling(1./127.5, offset=-1), # Scale [0, 255] -> [-1, 1]
    base_mn, layers.GlobalAveragePooling2D(), layers.Dense(128, activation='relu'), layers.Dense(len(class_names), activation='softmax')
])
sota_mn.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
t0=time.time(); sota_mn.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS_SOTA, verbose=1)
time_mn = (time.time() - t0)/EPOCHS_SOTA

# EfficientNetB0 (Needs raw [0, 255], has internal rescaling)
base_en = tf.keras.applications.EfficientNetB0(input_shape=(64, 64, 3), include_top=False, weights='imagenet')
base_en.trainable = False
sota_en = models.Sequential([
    layers.Input(shape=(64, 64, 3)),
    base_en, layers.GlobalAveragePooling2D(), layers.Dense(128, activation='relu'), layers.Dense(len(class_names), activation='softmax')
])
sota_en.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
t0=time.time(); sota_en.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS_SOTA, verbose=1)
time_en = (time.time() - t0)/EPOCHS_SOTA

sota_mn.save('sota_mn.h5'); size_mn = os.path.getsize('sota_mn.h5') / (1024*1024)
sota_en.save('sota_en.h5'); size_en = os.path.getsize('sota_en.h5') / (1024*1024)

preds_mn = np.argmax(sota_mn.predict(test_dataset, verbose=0), axis=1)
preds_en = np.argmax(sota_en.predict(test_dataset, verbose=0), axis=1)

print("="*50)
print(f"MOBILENET-V2 | Disk Size: {size_mn:.2f} MB | Avg Time/Epoch: {time_mn:.2f}s")
print(classification_report(test_labels, preds_mn, target_names=class_names))
print("="*50)
print(f"EFFICIENTNET-B0 | Disk Size: {size_en:.2f} MB | Avg Time/Epoch: {time_en:.2f}s")
print(classification_report(test_labels, preds_en, target_names=class_names))



In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Calculate exact accuracies (%)
acc_a = accuracy_score(test_labels, preds_a) * 100
acc_b = accuracy_score(test_labels, preds_b) * 100
acc_mn = accuracy_score(test_labels, preds_mn) * 100
acc_en = accuracy_score(test_labels, preds_en) * 100

# Standardize sizes to MB (Model A and B were in KB)
size_a_mb = size_a / 1024
size_b_mb = size_b / 1024

models_names = ['Model A', 'Model B', 'MobileNetV2', 'EfficientNetB0']
accuracies = [acc_a, acc_b, acc_mn, acc_en]
sizes_mb = [size_a_mb, size_b_mb, size_mn, size_en]
times = [time_a, time_b, time_mn, time_en]

print("\nGENERATING SUMMARY GRAPHS...")

# 1. Accuracy vs Model Size Scatter Plot
plt.figure(figsize=(10, 6))
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
for i in range(4):
    plt.scatter(sizes_mb[i], accuracies[i], color=colors[i], s=200, zorder=5, edgecolors='black')
    plt.annotate(models_names[i], (sizes_mb[i], accuracies[i]), xytext=(10, -5), textcoords='offset points', fontsize=12, fontweight='bold')

plt.title('Accuracy vs Model Size (Higher & Left is Better)', fontsize=15, fontweight='bold')
plt.xlabel('Model Size (MB)', fontsize=13)
plt.ylabel('Test Accuracy (%)', fontsize=13)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# 2. Average Training Time per Epoch Bar Chart
plt.figure(figsize=(10, 5))
bars = plt.bar(models_names, times, color=colors, alpha=0.8, edgecolor='black')
plt.title('Average Training Time per Epoch', fontsize=15, fontweight='bold')
plt.ylabel('Time (Seconds)', fontsize=13)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.1f} s", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


## 6. Final Comparison & Trade-off Discussion

| Metric | Custom Model B (Lightweight) | SOTA: MobileNetV2 | SOTA: EfficientNetB0 |
| :--- | :--- | :--- | :--- |
| **Total Parameters** | 79,168 (< 100k constraint) | ~2,350,000 | ~4,130,000 |
| **Model Size on Disk** | **0.99 MB** | 10.85 MB | 17.79 MB |
| **Time per Epoch** | 52.0 s | **33.1 s** | 63.6 s |
| **Test Accuracy** | 96.73% | 97.00% | **99.20%** |
| **Build Approach** | Built from scratch | Transfer Learning | Transfer Learning |

**Discussion & Strategic Trade-offs:**

- **Accuracy:** `EfficientNetB0` yields the highest accuracy (99.20%). However, `MobileNetV2` (97.00%) and our custom `Model B` (96.73%) follow very closely. The clearly defined macro-geometric features (length, width, ellipticity) of the rice grains mean that massive architectures are somewhat overkill for this specific classification task.

- **Memory Footprint:** This is where `Model B` heavily outshines the SOTA models. `Model B` strictly adheres to the sub-100k parameter budget, occupying just `0.99 MB`. SOTA architectures inherently consume larger memory profiles (`10.85 MB` to `17.79 MB`), rendering them impractical for deployment on low-memory edge IoT sensors like basic microcontrollers.

- **Computational Cost:** Transfer learning with `MobileNetV2` was surprisingly fast to train (~33s per epoch) because we freeze the base layers and only train the head. Training `Model B` from scratch takes slightly longer per epoch (~52s). However, during **inference** (deployment), `Model B` will execute significantly fewer Multiply-Accumulate (MAC) operations due to having 30x fewer parameters than MobileNet.

**Overall Conclusion:**
If the edge computational unit (such as a Raspberry Pi) has sufficient RAM and storage, employing SOTA transfer learning ensures maximum predictive accuracy and faster fine-tuning. However, for an ultra-low-resource hardware environment (e.g., an Arduino attached to an agricultural sorting machine), our **Depthwise Separable CNN (Model B)** serves as the definitive optimal choice delivering highly competitive classification metrics while strictly adhering to rigorous physical memory constraints.
